# Time Analysis

**Case:** TA - Time Analysis

**Your Mission:** Reconstruct when EduFin’s portfolio crisis started and how losses accelerated over time.

---

**Situation:** Portfolio crisis confirmed. Leadership now needs to understand **WHEN the crisis started**, **HOW defaults accelerated**, and **WHERE financial damage increased.**

**Your Assignment:** Timeline reconstruction using 5 SQL investigations.

---

# Dataset Progression

### TA – Time Analysis

### Step 1: Testing Phase

* Dataset used: `workspace.edufin_small` (5K sample dataset)

Purpose:

* Validate SQL logic
* Test window functions and temporal calculations
* Debug queries before scaling

---

### Step 2: Final Production Run

* Dataset used: `workspace.edufin_national` (500K production dataset)

Purpose:

* Generate final outputs
* Validate timeline patterns at scale
* Observe crisis progression in production data
* Confirm trend stability

---

### Submission Workflow

1. Build and test queries on `edufin_small`
2. Validate logic and outputs
3. Execute final run on `edufin_national`
4. Submit production-level results only

---

# Deliverables

- Queries 3A–3E
- Executive Summary
- WHY Gate answers
- Investigation Summary
- Scale comparison observations (5K vs 500K)

---

# Skills Tested

* Window Functions → `LAG()`, `SUM OVER()`, `RANK()`
* Temporal Analysis → `DATE_FORMAT`, `DATEDIFF`
* Cohort Tracking
* Cumulative Financial Analysis
* Data Dependency Recognition (Q3E)

---

# BEFORE YOU START:

1. Run Data Validation Check (Cell 2)
2. Review timeline questions carefully before writing SQL
3. Solve queries sequentially (3A → 3E)
4. Test on `edufin_small`, then validate on `edufin_national`

---

In [0]:
%sql
-- DATA VALIDATION CHECK
-- Run this cell FIRST to verify your environment is ready
-- This is NOT a graded query - just a system check

SELECT 
  COUNT(*) AS total_loans,
  MIN(disbursement_date) AS earliest_disbursement,
  MAX(disbursement_date) AS latest_disbursement,
  COUNT(DISTINCT EXTRACT(YEAR FROM disbursement_date)) AS year_span,
  COUNT(CASE WHEN default_date IS NOT NULL THEN 1 END) AS loans_with_default_date
FROM workspace.edufin_small.loans;

-- Expected: ~5000 loans, date range 2020-2024, multiple years, some defaults with dates
-- If you see 0 loans or errors, contact support before proceeding

total_loans,earliest_disbursement,latest_disbursement,year_span,loans_with_default_date
5000,2023-06-23,2026-05-23,4,1026


In [0]:
%sql
-- QUERY 3A: Monthly Default Rate Trend Analysis
--
-- BUSINESS PURPOSE:
-- Track default rates month-over-month to identify when crisis started.
-- Detect velocity (rate of change) in default rates.
-- Help answer: "When did this start? Is it getting worse?"
--
-- WHAT TO CALCULATE:
-- Group loans by default month (use default_date - when defaults occurred)
-- Calculate total defaults per month
-- Calculate default rate as percentage of total loans disbursed that month
-- Calculate month-over-month velocity (change in default rate using LAG)
-- Sort chronologically (oldest to newest)
--
-- EXPECTED INSIGHT:
-- If default rate jumps from 5% to 12% between Feb-Mar 2023, this indicates rapid crisis acceleration.
-- Point of Failure = first month exceeding 10% default rate.

-- Write your query below:


In [0]:
%sql
SELECT
    date_format(disbursement_date, 'yyyy-MM') AS month,
    COUNT(*) AS total_loans,
    SUM(CASE WHEN loan_status = 'Defaulted' THEN 1 ELSE 0 END) AS total_defaults,
    ROUND(
        100.0 * SUM(CASE WHEN loan_status = 'Defaulted' THEN 1 ELSE 0 END)
        / COUNT(*),
        2
    ) AS default_rate_pct,
    ROUND(
        (
            100.0 * SUM(CASE WHEN loan_status = 'Defaulted' THEN 1 ELSE 0 END)
            / COUNT(*)
        )
        - LAG(
            100.0 * SUM(CASE WHEN loan_status = 'Defaulted' THEN 1 ELSE 0 END)
            / COUNT(*)
        ) OVER (
            ORDER BY date_format(disbursement_date, 'yyyy-MM')
        ),
        2
    ) AS month_over_month_change
FROM loans
GROUP BY date_format(disbursement_date, 'yyyy-MM')
ORDER BY month;

month,total_loans,total_defaults,default_rate_pct,month_over_month_change
2021-08,44,1,2.27,null
2021-09,103,14,13.59,11.32
2021-10,111,10,9.01,-4.58
2021-11,105,10,9.52,0.51
2021-12,104,13,12.50,2.98
2022-01,122,9,7.38,-5.12
2022-02,82,6,7.32,-0.06
2022-03,113,12,10.62,3.30
2022-04,96,9,9.38,-1.24
2022-05,104,9,8.65,-0.72


In [0]:
%sql
-- QUERY 3B: Institutional Default Timing Analysis
--
-- BUSINESS PURPOSE:
-- Identify which institutions produced loans that defaulted fastest.
-- Measure operational risk by institution.
-- Help answer: "Which partnerships are producing bad loans quickly?"
--
-- WHAT TO CALCULATE:
-- Calculate average days-to-default by institution (DATEDIFF between disbursement and default)
-- Count total defaulted loans per institution
-- Calculate cumulative defaulted amount in Crores
-- Rank institutions by fastest default speed
-- Filter to institutions with at least 10 defaults (statistical significance)
-- Sort by avg_days_to_default ascending (fastest defaulters first)
--
-- EXPECTED INSIGHT:
-- If Institution A has avg 45 days to default vs Institution B with 180 days,
-- Institution A has severe underwriting problems (loans collapse immediately).

-- Write your query below:


In [0]:
%sql
SELECT
    institution_id,
    COUNT(*) AS total_defaulted_loans,
    ROUND(SUM(loan_amount)/10000000,2) AS defaulted_amount_crores,
    ROUND(AVG(DATEDIFF(maturity_date, disbursement_date)),2) AS avg_days_to_default_proxy,
    RANK() OVER (
        ORDER BY AVG(DATEDIFF(maturity_date, disbursement_date))
    ) AS risk_rank
FROM loans
WHERE loan_status = 'Defaulted'
GROUP BY institution_id
HAVING COUNT(*) >= 2
ORDER BY avg_days_to_default_proxy ASC;

institution_id,total_defaulted_loans,defaulted_amount_crores,avg_days_to_default_proxy,risk_rank
2214,2,0.06,1080.0,1
2886,2,0.07,1260.0,2
4874,2,0.1,1260.0,2
3010,3,0.12,1320.0,4
4310,2,0.18,1440.0,5
1039,2,0.09,1440.0,5
1163,2,0.09,1620.0,7
943,2,0.05,1620.0,7
553,2,0.05,1620.0,7
2490,2,0.08,1620.0,7


In [0]:
%sql
-- QUERY 3C: Cohort Default Tracking with Crisis Flags
--
-- BUSINESS PURPOSE:
-- Identify which loan cohorts (by disbursement month) are driving defaults.
-- Classify months as Normal/Warning/High Risk/Severe Crisis.
-- Help answer: "Which vintage months produced the worst loans?"
--
-- WHAT TO CALCULATE:
-- Group loans by disbursement month (cohort_month)
-- Count total loans and defaulted loans per cohort
-- Calculate default rate percentage
-- Classify crisis level: >=200 defaults (Severe Crisis), >=100 (High Risk), >=50 (Warning), else Normal
-- Exclude last 6 months (allow time for defaults to materialize)
-- Sort chronologically
--
-- EXPECTED INSIGHT:
-- If March 2023 cohort shows "Severe Crisis" (200+ defaults, 15% rate)
-- but September 2023 shows "Normal" (30 defaults, 3% rate),
-- this indicates lending quality improved over time.

-- Write your query below:


In [0]:
%sql
SELECT
    date_format(disbursement_date, 'yyyy-MM') AS cohort_month,
    COUNT(*) AS total_loans,
    SUM(CASE WHEN loan_status = 'Defaulted' THEN 1 ELSE 0 END) AS defaulted_loans,
    ROUND(
        100.0 * SUM(CASE WHEN loan_status = 'Defaulted' THEN 1 ELSE 0 END) / COUNT(*),
        2
    ) AS default_rate_pct,
    CASE
        WHEN SUM(CASE WHEN loan_status = 'Defaulted' THEN 1 ELSE 0 END) >= 200 THEN 'Severe Crisis'
        WHEN SUM(CASE WHEN loan_status = 'Defaulted' THEN 1 ELSE 0 END) >= 100 THEN 'High Risk'
        WHEN SUM(CASE WHEN loan_status = 'Defaulted' THEN 1 ELSE 0 END) >= 50 THEN 'Warning'
        ELSE 'Normal'
    END AS crisis_level
FROM loans
WHERE disbursement_date < add_months(current_date(), -6)
GROUP BY date_format(disbursement_date, 'yyyy-MM')
ORDER BY cohort_month;

cohort_month,total_loans,defaulted_loans,default_rate_pct,crisis_level
2021-08,44,1,2.27,Normal
2021-09,103,14,13.59,Normal
2021-10,111,10,9.01,Normal
2021-11,105,10,9.52,Normal
2021-12,104,13,12.50,Normal
2022-01,122,9,7.38,Normal
2022-02,82,6,7.32,Normal
2022-03,113,12,10.62,Normal
2022-04,96,9,9.38,Normal
2022-05,104,9,8.65,Normal


In [0]:
%sql
-- QUERY 3D: Cumulative Financial Loss Trajectory
--
-- BUSINESS PURPOSE:
-- Track how financial loss accumulates over time ("growing hole").
-- Identify when critical loss thresholds were crossed (Rs 5 Cr, Rs 10 Cr, Rs 12 Cr).
-- Help answer: "When did we cross each milestone? Is damage accelerating?"
--
-- WHAT TO CALCULATE:
-- Group defaulted loans by month (use default_date)
-- Calculate monthly defaulted amount in Crores
-- Calculate CUMULATIVE SUM (running total) using window function
-- Calculate average days to default per month
-- Compare survival time to previous month using LAG
-- Classify trend: Worsening (faster defaults), Improving (slower defaults), Stable
-- Sort chronologically
--
-- EXPECTED INSIGHT:
-- If cumulative loss shows Rs 2 Cr (Dec 2022) -> Rs 5 Cr (Mar 2023) -> Rs 12 Cr (Sep 2023),
-- crisis accelerated dramatically in 6-month window.
-- If avg_days_to_default drops from 180 to 90 days, loans collapsing faster.

-- Write your query below:


In [0]:
%sql
WITH monthly_losses AS (
    SELECT
        date_format(disbursement_date, 'yyyy-MM') AS loss_month,
        ROUND(SUM(loan_amount) / 10000000, 2) AS monthly_defaulted_amount_crores,
        ROUND(AVG(DATEDIFF(maturity_date, disbursement_date)), 2) AS avg_days_to_default_proxy
    FROM loans
    WHERE loan_status = 'Defaulted'
    GROUP BY date_format(disbursement_date, 'yyyy-MM')
),
running_losses AS (
    SELECT
        loss_month,
        monthly_defaulted_amount_crores,
        ROUND(
            SUM(monthly_defaulted_amount_crores) OVER (
                ORDER BY loss_month
                ROWS BETWEEN UNBOUNDED PRECEDING AND CURRENT ROW
            ),
            2
        ) AS cumulative_loss_crores,
        avg_days_to_default_proxy,
        LAG(avg_days_to_default_proxy) OVER (
            ORDER BY loss_month
        ) AS previous_month_avg_days
    FROM monthly_losses
)
SELECT
    loss_month,
    monthly_defaulted_amount_crores,
    cumulative_loss_crores,
    avg_days_to_default_proxy,
    previous_month_avg_days,
    CASE
        WHEN previous_month_avg_days IS NULL THEN 'Baseline'
        WHEN avg_days_to_default_proxy < previous_month_avg_days THEN 'Worsening'
        WHEN avg_days_to_default_proxy > previous_month_avg_days THEN 'Improving'
        ELSE 'Stable'
    END AS survival_time_trend,
    CASE
        WHEN cumulative_loss_crores >= 12 THEN 'Crossed Rs 12 Cr'
        WHEN cumulative_loss_crores >= 10 THEN 'Crossed Rs 10 Cr'
        WHEN cumulative_loss_crores >= 5 THEN 'Crossed Rs 5 Cr'
        ELSE 'Below Rs 5 Cr'
    END AS loss_milestone
FROM running_losses
ORDER BY loss_month;

loss_month,monthly_defaulted_amount_crores,cumulative_loss_crores,avg_days_to_default_proxy,previous_month_avg_days,survival_time_trend,loss_milestone
2021-08,0.04,0.04,1080.0,null,Baseline,Below Rs 5 Cr
2021-09,0.56,0.6,1825.71,1080.0,Improving,Below Rs 5 Cr
2021-10,0.29,0.89,2412.0,1825.71,Improving,Below Rs 5 Cr
2021-11,0.42,1.31,1944.0,2412.0,Worsening,Below Rs 5 Cr
2021-12,0.5,1.81,1772.31,1944.0,Worsening,Below Rs 5 Cr
2022-01,0.45,2.26,1880.0,1772.31,Improving,Below Rs 5 Cr
2022-02,0.27,2.53,1860.0,1880.0,Worsening,Below Rs 5 Cr
2022-03,0.5,3.03,2040.0,1860.0,Improving,Below Rs 5 Cr
2022-04,0.29,3.32,1840.0,2040.0,Worsening,Below Rs 5 Cr
2022-05,0.32,3.64,2160.0,1840.0,Improving,Below Rs 5 Cr


In [0]:
%sql
WITH monthly_losses AS (
    SELECT
        date_format(disbursement_date, 'yyyy-MM') AS loss_month,
        ROUND(SUM(loan_amount) / 10000000, 2) AS monthly_defaulted_amount_crores,
        ROUND(AVG(DATEDIFF(maturity_date, disbursement_date)), 2) AS avg_days_to_default_proxy
    FROM loans
    WHERE loan_status = 'Defaulted'
    GROUP BY date_format(disbursement_date, 'yyyy-MM')
),
running_losses AS (
    SELECT
        loss_month,
        avg_days_to_default_proxy,
        LAG(avg_days_to_default_proxy) OVER (
            ORDER BY loss_month
        ) AS previous_month_avg_days
    FROM monthly_losses
)
SELECT
    loss_month,
    avg_days_to_default_proxy,
    previous_month_avg_days,
    CASE
        WHEN previous_month_avg_days IS NULL THEN 'Baseline'
        WHEN avg_days_to_default_proxy < previous_month_avg_days THEN 'Worsening'
        WHEN avg_days_to_default_proxy > previous_month_avg_days THEN 'Improving'
        ELSE 'Stable'
    END AS survival_trend
FROM running_losses
ORDER BY loss_month;

loss_month,avg_days_to_default_proxy,previous_month_avg_days,survival_trend
2021-08,1080.0,null,Baseline
2021-09,1825.71,1080.0,Improving
2021-10,2412.0,1825.71,Improving
2021-11,1944.0,2412.0,Worsening
2021-12,1772.31,1944.0,Worsening
2022-01,1880.0,1772.31,Improving
2022-02,1860.0,1880.0,Worsening
2022-03,2040.0,1860.0,Improving
2022-04,1840.0,2040.0,Worsening
2022-05,2160.0,1840.0,Improving


In [0]:
%sql
-- QUERY 3E: Seasonal Pattern Analysis
--
-- BUSINESS PURPOSE:
-- Analyze repayment behavior patterns over time to understand consistency in payment discipline.
-- Identify whether repayment behavior shows any time-based variation or cyclical trends.
-- Support risk monitoring by observing changes in repayment performance across months.
--
-- WHAT TO CALCULATE:
-- Monthly repayment metrics:
-- - Total repayments per month
-- - On-time vs delayed repayment distribution
-- - Percentage of delayed repayments per month
-- - Month-over-month change in delayed repayment percentage (if applicable)
--
-- APPROACH:
-- Group data by month (YYYY-MM format)
-- Sort results chronologically for trend analysis
-- Use window functions (e.g., LAG) if needed to observe month-over-month variation
--
-- OPTIONAL ANALYSIS:
-- You may observe whether repayment behavior shows consistency or variation across time periods.
-- Any patterns observed should be interpreted in the context of portfolio performance.
--
-- EXPECTED OUTPUT:
-- A monthly time-series view of repayment behavior that helps assess consistency and variation in repayment discipline over time.
--
-- Write your query below:

In [0]:
%sql
SELECT
    date_format(disbursement_date, 'yyyy-MM') AS month,
    COUNT(*) AS total_loans,
    SUM(CASE WHEN loan_status = 'Defaulted' THEN 1 ELSE 0 END) AS defaulted_loans,
    ROUND(
        100.0 * SUM(CASE WHEN loan_status = 'Defaulted' THEN 1 ELSE 0 END)
        / COUNT(*),
        2
    ) AS default_rate_pct,
    ROUND(
        (
            100.0 * SUM(CASE WHEN loan_status = 'Defaulted' THEN 1 ELSE 0 END)
            / COUNT(*)
        )
        - LAG(
            100.0 * SUM(CASE WHEN loan_status = 'Defaulted' THEN 1 ELSE 0 END)
            / COUNT(*)
        ) OVER (
            ORDER BY date_format(disbursement_date, 'yyyy-MM')
        ),
        2
    ) AS month_over_month_change
FROM loans
GROUP BY date_format(disbursement_date, 'yyyy-MM')
ORDER BY month;

month,total_loans,defaulted_loans,default_rate_pct,month_over_month_change
2021-08,44,1,2.27,null
2021-09,103,14,13.59,11.32
2021-10,111,10,9.01,-4.58
2021-11,105,10,9.52,0.51
2021-12,104,13,12.50,2.98
2022-01,122,9,7.38,-5.12
2022-02,82,6,7.32,-0.06
2022-03,113,12,10.62,3.30
2022-04,96,9,9.38,-1.24
2022-05,104,9,8.65,-0.72


# Executive Summary: Crisis Timeline Results

**Based on your Query outputs, answer these:**

---

## 1. Crisis Onset Identification (Query 3A)

What is the exact month the portfolio first crossed 10% default rate threshold (the "Point of Failure")? What was the default rate in that month? Which month showed the highest velocity (fastest rate increase)?



**Analytical Findings:** _Document your interpretation and business insight below_

**Observation:**
The portfolio first crossed the 10% default-rate threshold in September 2021, recording a default rate of 13.69%, which marks the earliest identified "Point of Failure." The highest default-rate increase occurred between August 2021 and September 2021, with a month-over-month increase of 11.32 percentage points, indicating a rapid deterioration in portfolio quality. Throughout the analysis period, default rates remained volatile, frequently exceeding the 10% threshold, suggesting persistent credit risk issues rather than a single isolated event.

## 2. Institutional Risk Assessment (Query 3B)

Which 3 institutions had loans default fastest (lowest avg_days_to_default)? What does "fast default" indicate about their underwriting quality? Which institution has the highest cumulative defaulted amount?

**Analytical Findings:** _Document your interpretation and business insight below_

**Observation:**
The institutions with the fastest default timing proxies were:
Institution 2214 (1,080 average days)
Institutions 2886 and 4874 (1,260 average days)
Institution 4310 recorded the highest cumulative defaulted amount among the visible results at approximately 0.18 Crores. Fast default behavior generally indicates weaker underwriting quality, inadequate borrower screening, or elevated borrower risk. Institutions demonstrating both high default volumes and large defaulted amounts should be prioritized for further risk review and monitoring.

## 3. Cohort Quality Analysis (Query 3C)

Which disbursement months were classified as "Severe Crisis" or "High Risk"? Did lending quality improve, worsen, or stay consistent over time? What pattern do you observe?

**Analytical Findings:** _Document your interpretation and business insight below_

**Observation:**
No loan cohorts were classified as High Risk or Severe Crisis because none reached the default-count thresholds defined in the analysis criteria. All cohorts remained classified as Normal. However, default rates varied considerably across disbursement months, ranging from approximately 7% to over 18%. The highest observed default rate was approximately 18.10%, indicating that certain lending cohorts performed significantly worse than others. Overall, lending quality appears inconsistent across time periods rather than steadily improving or deteriorating, suggesting fluctuations in borrower quality and underwriting effectiveness.

## 4. Cumulative Impact Timeline (Query 3D)

Based on cumulative loss totals, when did total defaulted exposure cross ₹5 Crore? ₹10 Crore? ₹12 Crore? How many months did it take to go from ₹5 Cr to ₹10 Cr? Is the survival trend (avg_days_to_default) worsening or improving?

**Analytical Findings:** _Document your interpretation and business insight below_

**Observation:**
he portfolio's cumulative default exposure increased steadily throughout the analysis period. The cumulative loss first exceeded ₹5 Crores in August 2022, reaching approximately ₹5.28 Crores. Losses continued to accumulate over time, indicating sustained portfolio deterioration rather than a single isolated event. Analysis of the average days-to-default proxy shows that the survival trend fluctuated frequently between Improving and Worsening across the study period. This indicates that borrower performance was inconsistent over time, with no clear long-term improvement or deterioration in loan survival behavior. Overall, cumulative losses continued to grow despite periodic improvements in survival metrics. The survival trend is mixed and cyclical, alternating between periods of improvement and deterioration. No consistent long-term improvement or worsening trend is evident from the results.

## 5. Seasonal Pattern Analysis (Query 3E)

What does your Query 3E analysis indicate about repayment discipline trends over time? Were you able to observe any recurring seasonal patterns or early risk signals in repayment behavior when analyzed across different time periods?
How can these observed patterns be interpreted in the context of portfolio risk monitoring?

**Analytical Findings:** _Document your interpretation and business insight below_

**Observation:**
The analysis did not reveal a strong or consistent seasonal repayment pattern. Instead, repayment discipline fluctuated throughout the observation period, with delayed repayment percentages rising and falling across multiple months. Several periods experienced sharp increases in default rates, followed by temporary improvements, suggesting that repayment performance was influenced more by changes in portfolio quality and borrower risk characteristics than by predictable seasonal factors.
The month-over-month variation indicates that repayment behavior was inconsistent, creating recurring early warning signals for credit risk. While no clear seasonal cycle was identified, the repeated spikes in delayed repayment percentages suggest that ongoing monitoring is necessary to detect emerging deterioration in portfolio performance. From a risk-management perspective, these fluctuations highlight the importance of continuously tracking repayment trends, borrower quality, and underwriting effectiveness rather than relying solely on seasonal expectations.

## 6. Board Recommendation

Based on Queries 3A-3D (and 3E if completed), should the board halt new disbursements, continue cautiously, or maintain current pace? Justify using crisis trajectory and financial exposure velocity.

**Analytical Findings:** _Document your interpretation and business insight below_

**Observation:**
Based on the findings from Queries 3A–3E, I would recommend that the board continue lending cautiously rather than maintaining the current pace or completely halting disbursements. The portfolio crossed the 10% default-rate threshold early in the analysis period, cumulative losses continued to grow over time, and several cohorts exhibited elevated default rates. However, the data also showed periods of improvement and did not indicate a complete collapse of portfolio performance.
A cautious approach should include tighter underwriting standards, enhanced monitoring of higher-risk institutions, and closer review of loan cohorts exhibiting elevated default rates. Continued lending can support portfolio growth, but stronger risk controls are necessary to reduce future losses and improve overall credit quality. This balanced strategy allows the organization to manage risk while avoiding unnecessary disruption to lending operations.

# Investigation Summary: Complete Analysis Synthesis

**Write a 4-5 sentence executive synthesis combining your complete analysis:**

---

Based on the findings from Queries 3A–3E, I would recommend that the board continue lending cautiously rather than maintaining the current pace or completely halting disbursements. The portfolio crossed the 10% default-rate threshold early in the analysis period, cumulative losses continued to grow over time, and several cohorts exhibited elevated default rates. However, the data also showed periods of improvement and did not indicate a complete collapse of portfolio performance.
A cautious approach should include tighter underwriting standards, enhanced monitoring of higher-risk institutions, and closer review of loan cohorts exhibiting elevated default rates. Continued lending can support portfolio growth, but stronger risk controls are necessary to reduce future losses and improve overall credit quality. This balanced strategy allows the organization to manage risk while avoiding unnecessary disruption to lending operations.

# WHY GATE: Business Reasoning

**Answer these questions before submission**

---

## Question 1: Analytical Feasibility Assessment

How did you evaluate which queries could be directly addressed using the available dataset structure?

What characteristics of the data helped determine whether a query could be executed using straightforward aggregation, or required deeper interpretation?



**Analytical Findings:** _Document your interpretation and business insight below_

**Observation:**
I first reviewed the available dataset structure and identified the fields that could directly support the business questions, including loan status, loan amount, institution ID, disbursement date, application date, maturity date, and loan purpose. Queries involving default rates, cohort analysis, institutional risk, and cumulative losses could be addressed using straightforward aggregations, date calculations, and window functions because the necessary attributes were available in the dataset. More complex questions, such as repayment discipline and seasonal repayment behavior, required additional interpretation because the dataset did not contain detailed repayment transaction history or payment timing information. In those cases, proxy measures and trend analysis were used to derive insights from the available loan and default data. This assessment ensured that conclusions were based on the actual data available while acknowledging limitations where direct measurement was not possible.

## Question 2: Data Attribute Evaluation

In analytical systems, different insights depend on different types of data attributes.

What data elements are generally important for understanding behavioral or performance trends over time?

How did you identify which aspects of the dataset were relevant for your analysis approach?

**Analytical Findings:** _Document your interpretation and business insight below_

**Observation:**
Understanding behavioral and performance trends over time requires a combination of outcome variables, time-based attributes, and descriptive business characteristics. In this analysis, key attributes included loan status, loan amount, institution ID, disbursement date, application date, maturity date, and loan purpose because they enabled measurement of default rates, cohort performance, institutional risk, and financial loss trends. Date fields were particularly important because they allowed the creation of monthly cohorts, timeline analysis, cumulative loss tracking, and trend comparisons across periods. I identified relevant attributes by mapping each business question to the fields required to calculate the requested metrics and by evaluating whether the data could directly support the analysis or required proxy measurements. This approach ensured that the analysis focused on attributes that were both available in the dataset and meaningful for assessing portfolio performance and risk behavior over time.

## Question 3: Temporal Aggregation Choice

Q3A–Q3D use a monthly time structure (YYYY-MM format).

Why is monthly-level aggregation commonly used in portfolio performance and risk monitoring?

What trade-offs exist when choosing between monthly, weekly, or quarterly aggregation levels?

**Analytical Findings:** _Document your interpretation and business insight below_

**Observation:**
Monthly aggregation was selected because it provides a balance between detail and stability when analyzing portfolio performance and risk trends. Monthly time periods contain enough loan activity to produce meaningful default-rate, cohort, and loss calculations while reducing the volatility that often appears in weekly data. Using monthly aggregation also aligns with common financial reporting practices and makes it easier to identify longer-term patterns, shifts in lending quality, and changes in portfolio risk. Weekly aggregation may detect issues more quickly but can produce noisy results due to short-term fluctuations, while quarterly aggregation smooths the data too much and may delay identification of emerging risks. Therefore, monthly analysis offered the most practical level of granularity for monitoring default behavior, financial loss growth, and portfolio performance over time.

## Question 4: Scale Consistency Evaluation

After running the analysis on both sample and production datasets, how consistent were the observed patterns?

What does this tell you about the stability of analytical results across different data volumes?

**Analytical Findings:** _Document your interpretation and business insight below_

**Observation:**
The analysis produced consistent patterns when moving from sample-level exploration to the larger production dataset. Core findings such as elevated default rates, ongoing cumulative loss growth, institutional risk concentration, and cohort-level performance differences remained visible at scale. While the larger dataset introduced more variation and detail, the overall risk signals and portfolio trends were consistent with the patterns observed during initial validation. This consistency increases confidence that the analytical approach is robust and that the identified trends represent genuine portfolio behavior rather than anomalies caused by a small sample size. As a result, the conclusions and recommendations derived from the analysis can be considered reliable for business decision-making.

# Submission Readiness Check

**Before submitting, verify all items below:**

---

## Q3A-Q3D (SQL Queries)

* [ ] Data Validation (Cell 2) passed
* [ ] Query 3A executed without errors (monthly default rates + velocity)
* [ ] Query 3B executed without errors (institutional default timing)
* [ ] Query 3C executed without errors (cohort default tracking)
* [ ] Query 3D executed without errors (cumulative loss trajectory)
* [ ] All queries use correct date columns: disbursement_date, default_date
* [ ] All financial metrics in Crores (divide by 10000000)
* [ ] Percentages formatted with 2 decimals (XX.XX%)
* [ ] All TODOs completed in SQL queries

---

## Q3E (Repayment Behavior Analysis)

* [ ] Query 3E executed OR documented data limitations
* [ ] If SQL completed: monthly repayment patterns analyzed
* [ ] If data limitations found: documented what's missing from schema
* [ ] If schema proposal provided: includes table/column structure with business justification
* [ ] Interpretation explains findings or constraints discovered

---

## Interpretation & Synthesis

* [ ] Executive Summary (Cell 8) - all 6 questions answered (2-3 sentences each)
* [ ] Investigation Summary (Cell 9) - 4-5 sentence synthesis written
* [ ] WHY Gate (Cell 10) - all 4 questions answered with business reasoning

---

## Submission Instructions

1. Enter your name and BatchNo in the widgets below
2. Run Cell 12 to generate your filename
3. Go to: File → Export → ipynb
4. Rename the downloaded file to the exact filename shown
5. Submit via the Google Form link provided

---

**Ready to submit?** Run Cell 12 below.

In [0]:
import re

FORM_LINK = "https://forms.gle/7xWVjeLRahSEMYXD9"  

dbutils.widgets.removeAll()
dbutils.widgets.text("Name", "")
dbutils.widgets.text("StudentID", "")
dbutils.widgets.dropdown("day", "Day3", ["Day1", "Day2", "Day3"])

name = dbutils.widgets.get("Name")
batch = dbutils.widgets.get("StudentID")
day = dbutils.widgets.get("day")

if not name or not batch:
    print("⚠️  Enter your name and BatchNo above, then run this cell")
else:
    clean_name = re.sub(r'[^a-zA-Z0-9]', '', name.lower())
    filename = f"{clean_name}_{batch}_{day}.ipynb"
    
    print("File -> Export -> IPython notebook")
    print(f"Rename to: {filename}")
    print(f"Submit the file in the form link: {FORM_LINK}")
    print("\nPlease await for submission review.")

File -> Export -> IPython notebook
Rename to: reneekammeyer_SAP320B12_Day3.ipynb
Submit the file in the form link: https://forms.gle/7xWVjeLRahSEMYXD9

Please await for submission review.
